In [4]:
import json
from neo4j import GraphDatabase

cpid_filename = "PubChemAnnotations_Consumer-Product-Information-Database-CPID_Household-Products-Compound.json"

with open(cpid_filename, encoding="utf-8") as fp:
    d = json.load(fp)

annotations = d["Annotations"]["Annotation"]

In [ ]:
from neo4j import GraphDatabase
import json

with open('neo4j_dbinfo', 'r') as f:
    neo4j_info = json.load(f)


uri = neo4j_info["uri"]
user = neo4j_info["username"]
password = neo4j_info["password"]

driver = GraphDatabase.driver(
    uri,
    auth=(user, password)
)

In [ ]:
import re
import time
from neo4j import GraphDatabase
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

options = Options()
options.add_argument("--headless=new")
web_driver = webdriver.Chrome(options=options)

def create_cpid_node(tx, annotation):

    data = annotation["Data"][0]

    data_value = "\n".join(
        item["String"] for item in data["Value"]["StringWithMarkup"]
    )

    linked_records = annotation.get("LinkedRecords", {})
    name=annotation.get("Name")
    
    casrn = ""
    url = None
    dtxsid = ""
    try:

        url = annotation["Data"][0]["Value"]["StringWithMarkup"][0]["Markup"][0]["URL"]
        web_driver.get(url)

        time.sleep(3)        

        labels = web_driver.find_elements(By.XPATH, "//*[contains(normalize-space(.), 'CAS')]")

        for label in labels:
            try:
                container = label.find_element(By.XPATH, "./ancestor::*[self::div or self::section][1]")
                m = re.search(r"\b\d{2,7}-\d{2}-\d\b", container.text)
                if m:
                    casrn = m.group(0)
                    break
            except:
                pass

            elem = web_driver.find_element(
                By.XPATH,
                "//a[starts-with(@href, 'https://comptox.epa.gov/dashboard/')]"
            )

            href = elem.get_attribute("href")

            if href and href.startswith("https://comptox.epa.gov/dashboard/"):
                m = re.search(r"(DTXSID\d+)", href)
                if m:
                    dtxsid = m.group(1)  

    except:
        pass

    print(name, casrn, dtxsid)

    query = """
    CREATE (c:CPID {
        SourceName: $SourceName,
        SourceID: $SourceID,
        Name: $Name,
        Description: $Description,
        URL: $URL,
        LicenseNote: $LicenseNote,
        LicenseURL: $LicenseURL,
        ANID: $ANID,

        `Data.type`: $DataType,
        `Data.#TOCHeading`: $DataTOCHeading,
        `Data.Name`: $DataName,
        `Data.Value`: $DataValue,

        `LinkedRecords.CID`: $LinkedRecordsCID,
        `LinkedRecords.RefChemID`: $LinkedRecordsRefChemID,
        `casrn`: $casrn,
        `dtxsid`: $dtxsid,
        `pubchem_url`: $pubchem_url
    })
    """

    tx.run(
        query,
        SourceName=annotation.get("SourceName"),
        SourceID=annotation.get("SourceID"),
        Name=annotation.get("Name"),
        Description=annotation.get("Description"),
        URL=annotation.get("URL"),
        LicenseNote=annotation.get("LicenseNote"),
        LicenseURL=annotation.get("LicenseURL"),
        ANID=annotation.get("ANID"),

        DataType=data["TOCHeading"]["type"],
        DataTOCHeading=data["TOCHeading"]["#TOCHeading"],
        DataName=data["Name"],
        DataValue=data_value,

        LinkedRecordsCID=str(linked_records.get("CID", [])),
        LinkedRecordsRefChemID=str(linked_records.get("RefChemID", [])),
        casrn=casrn,
        dtxsid=dtxsid,
        pubchem_url=url
    )

with driver.session() as session:
    for annotation in annotations:
        try:
            session.execute_write(create_cpid_node, annotation)
        except:
            print(annotation)

driver.close()
web_driver.quit()

print("CPID nodes created.")

Formaldehyde 50-00-0 DTXSID7020637
Ergocalciferol 50-14-6 DTXSID5020233
Lactic acid 50-21-5 DTXSID7023192
Hydrocortisone 50-23-7 DTXSID7020714
Benzo(a)pyrene 50-32-8 DTXSID2020139
Chloroquine phosphate 50-63-5 DTXSID7044681
Sorbitol 50-70-4 DTXSID5023588
Ascorbic Acid (Vitamin C) 50-81-7 DTXSID5020106
Glucose 50-99-7 DTXSID501015215
Piperonyl butoxide 51-03-6 DTXSID1021166
2-Bromo-2-nitropropane-1,3-diol 52-51-7 DTXSID8024652
Trichlorfon 52-68-6 DTXSID0021389
Cysteine Hcl 52-89-1 DTXSID0020367
Nicotine 54-11-5 DTXSID1020930
Sodium Salicylate 54-21-7 DTXSID5021708
p-Methylaminophenol sulfate 55-55-0 DTXSID6025565
Isoflurophate [USP] 55-91-4 DTXSID1040667
Cystamine dihydrochloride 56-17-7 DTXSID6058766
Carbon tetrachloride 56-23-5 DTXSID8020250
bis(Tributyltin) oxide 56-35-9 DTXSID9020166
Glycine 56-40-6 DTXSID9020667
Serine 56-45-1 DTXSID301031857
Glycerin 56-81-5 DTXSID9020663
Glutamine 56-85-9 DTXSID1023100
Glutamic Acid 56-86-0 DTXSID5020659
Lysine 56-87-1 DTXSID6023232
Chlorhexidine

In [ ]:
# MATCH (c:CPID)
# MATCH (a:Chemical)
# WHERE a.dtxsid = c.dtxsid
# MERGE (c)-[:HAS_CHEMICAL]->(a);